<img src="https://cads.ai/image/CADSLogo.png" style="height: 100px;" align=left>
<div align=right><svg height=100px viewBox="0 0 24 24" fill="none" xmlns="http://www.w3.org/2000/svg"><g clip-path="url(#clip1_4783_495)"><path d="M6.64803 14.103C7.89438 12.8566 8.595 11.1643 8.595 9.40177C8.595 7.63928 7.89378 5.94697 6.64803 4.70058L1.94697 0C0.701225 1.24639 0 2.9387 0 4.70119C0 6.46368 0.701225 8.15599 1.94697 9.40238L6.64742 14.103H6.64803Z" fill="#7B9AB5"></path><path d="M16.4845 14.5379C15.2388 13.2921 13.5459 12.5908 11.7841 12.5908C10.0222 12.5908 8.32936 13.2921 7.08301 14.5379L11.7841 19.239C13.0298 20.4848 14.7227 21.1861 16.4851 21.1861C18.2476 21.1861 19.9398 20.4848 21.1862 19.239L16.4851 14.5379H16.4845Z" fill="#7B9AB5"></path><path d="M1.95832 19.228C3.20468 20.4738 4.89693 21.1751 6.65938 21.1751V14.5269H0.0107422C0.0113472 16.2893 0.711968 17.9817 1.95832 19.228Z" fill="#7B9AB5"></path><path d="M18.2997 7.58717C17.0533 6.34138 15.3611 5.63953 13.598 5.64014C11.8356 5.64014 10.1433 6.34138 8.89697 7.58777L13.598 12.289L18.2997 7.58717Z" fill="#7B9AB5"></path></g></svg></div>

<a box-sizing="border-box" color-scheme=dark font-family="IBM Plex Sans" line-height=30px font-size=72px font-weight=700 align=center>© 2026 — Mehdi Lotfinejad ⨯ Azhar bin Ab Halim @ DATATWEETS</a>

# Meaning Map and Brief
Day 1 | Notebook 4 of 4

![A street map with a river and a park. An orange diamond shows a question about a taxi claim. Inside a dashed circle labelled near are three policy pins; one of them, overtime meals, is flagged as a wrong neighbour because it shares words but not meaning. Other policy pins sit far outside the circle.](media/d1-04-hero-meaning-map.svg)

**You can re-run any practice until it works. Nothing here is recorded or marked.**

**Today in one sentence:** you find the nearest policy by meaning, not only by shared words, and write a short checked brief.

## Today's story

Seri Maju Berhad (fictional; Kuala Lumpur, Penang, Johor Bahru) has ten short policies. A helpdesk helper must find the *right* policy for a question before it answers. You are to build a **meaning map** that finds the nearest policy note, and you learn why the nearest note is sometimes wrong. Next, the Corporate Affairs team asks for a short, checked brief about TV audiences for a staff watch-party, and you use everything: a written definition of done, a structured answer, one repair and a call budget.

**Fictional example, not legal or tax advice.*

## What you will be able to do

6. Turn text into vectors, find the nearest policy note, and explain one wrong neighbour.
7. Run a checked brief: done-definition, structured output, one repair, a stop budget, and a human read.

---

**Prerequisites:** `d1-01-*.ipynb` to `d1-03-*.ipynb` help. Needs `scikit-learn`.

**Words you do not know?** See the [Glossary](glossary.md).

In [1]:
# Answer switch (change it, then choose Run > Restart Kernel and Run All Cells).
#     False = you type the blanks.
#     True = Run All reads the reference answers.
USE_SOLUTIONS = globals().get("USE_SOLUTIONS", False)

In [2]:
%pip install "langchain-groq>=0.3,<0.4"

In [3]:
# ---- CANONICAL SETUP ----
# ---- HELPER - Run it, then ignore it. ----


## You can re-run any practice until it works. Nothing here is recorded or marked.
# Standard-library helpers for settings files, timing and folder paths.
import json
import os
import sys
import time
from glob import glob
from pathlib import Path
from typing import Literal

# Third-party pieces: HTTP client, Groq chat model, and LangChain base classes for models and embeddings.
import httpx
from dotenv import load_dotenv
from langchain_core.caches import InMemoryCache
from langchain_core.embeddings import Embeddings
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_groq import ChatGroq

# scikit-learn powers the offline TF-IDF embeddings; stop early with an install hint if it is missing.
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
except ImportError as e:
    raise ImportError(
        "scikit-learn is needed for TfidfEmbeddings. In a terminal run: "
        "pip install scikit-learn  (then restart the kernel)."
    ) from e

# Load a default .env file, if there is one, into the environment.
load_dotenv()


SOLUTIONS = {}  # step_id -> (builder function, one-sentence why, code text shown by show_solution)
CHECKPOINTS = {}  # part_id -> function(ns) that rebuilds the variables a late learner needs (canned/mock only)
CODES = {}  # part -> reference code texts, in order

# Look up per-million-token prices for the chosen GROQ_MODEL, used for cost estimates.
# MYR is preferred, then USD, then a built-in default.
with open("_groq_model_per_million_prices.json", mode="rb") as _model_prices_json:
    _MODEL_PRICES = json.load(_model_prices_json).get(
        os.environ.get("GROQ_MODEL", ""), {}
    )
    model_input_price = (_model_input_price := _MODEL_PRICES.get("input", {})).get(
        "MYR", _model_input_price.get("USD", 0.50)
    )
    model_output_price = (_model_output_price := _MODEL_PRICES.get("output", {})).get(
        "MYR", _model_output_price.get("USD", 2.00)
    )


def register(step_id, why, code=""):
    """Decorator used by the notebook: SOLUTIONS['d1-01-p2'] = the reference builder."""

    def deco(fn):
        SOLUTIONS[step_id] = (fn, why, code or getattr(fn, "__doc__", "") or "")
        return fn

    return deco


def checkpoint(part_id):
    """Decorator: register the function that rebuilds this Part's variables (see fast_forward)."""

    def deco(fn):
        CHECKPOINTS[part_id] = fn
        return fn

    return deco


def pick(step_id, attempt=None, is_using_solutions: bool = True):
    """Use your attempt, or the reference answer if USE_SOLUTIONS is True or your attempt is None."""
    if is_using_solutions or attempt is None:
        if step_id not in SOLUTIONS:
            raise KeyError(
                f"No reference answer registered for '{step_id}'. Click the heading 'Part A', "
                "choose Run > Run All Above Selected Cell, then run this cell again."
            )
        return SOLUTIONS[step_id][0]()
    return attempt


def check(step_id, ok, hint=""):
    """Plain self-check: tick or cross plus a hint. Unlimited re-runs. Nothing is stored or counted."""
    print(
        ("\u2714 " + step_id + ": looks right")
        if ok
        else (
            "\u2718 "
            + step_id
            + ": not yet. "
            + (hint or "Re-read the step, change your code, run again.")
        )
    )
    return bool(ok)


def show_solution(step_id):
    """Print the reference code and why it works. Keyboard-friendly: no copying needed, use pick()."""
    if step_id not in SOLUTIONS:
        print("No reference answer registered for", step_id)
        return
    _, why, code = SOLUTIONS[step_id]
    print(code.rstrip() or "(see the reference cell)")
    print("Why:", why)


def fast_forward(part_id):
    """Late or stuck? Rebuild this Part's variables from reference code (canned/mock, no API calls)."""

    if part_id not in CHECKPOINTS:
        print(
            "Not ready yet. Click the heading 'Part A', choose Run > Run All Above Selected Cell, "
            "then run this cell again. Known parts:",
            ", ".join(sorted(CHECKPOINTS)) or "(none registered yet)",
        )
        return

    if globals().get("LLM_MODE") == "groq":
        print("Note: fast-forward uses canned replies, not the live model.")

    CHECKPOINTS[part_id](globals())
    print("\u2714 Caught up to", part_id, "- continue with the next cell.")


# Load the workplace policy texts as system messages so every answer can be grounded in them.
def get_system_corpus(context: Literal["workplace"]) -> list[tuple[str, str]]:
    _corpus_dir: Path
    system_prompt: list[tuple[str, str]] = []

    if context == "workplace":
        _corpus_dir = Path.cwd() / "data" / "workplace*" / "*.txt"

    else:
        raise ValueError(f"Unsupported corpus: '{context}'")

    # Read each policy .txt file under data/workplace*/ as one system message.
    for _corpus_pathname in glob(str(_corpus_dir)):
        with open(_corpus_pathname, "r", encoding="utf-8") as _corpus_file:
            system_prompt.append(("system", _corpus_file.read()))

    # Finish with a rule that makes answers cite their source identifier.
    system_prompt.append(
        (
            "system",
            "Assistant should always cite source identifier to support claim when answering any asks.",
        )
    )

    return system_prompt


# Remember a code snippet under its Part so a late learner can rebuild that Part.
def state(part, code):
    CODES.setdefault(part, []).append(code.strip())


# Register a reference answer: keep its code text and a builder that runs it and returns result.
def ref(part, step, why, code):
    code = code.strip()
    if part:
        CODES.setdefault(part, []).append(code)

    # Run the reference code in a copy of the notebook variables and hand back its result.
    def build():
        ns = dict(globals())
        exec(code, ns)
        return ns["result"]

    register(step, why, code)(build)


# Define how to catch up: re-run the stored code of the listed Parts in order (used by fast_forward).
def rebuild(part, *parts):
    checkpoint(part)(lambda ns: [exec(c, ns) for p in parts for c in CODES[p]])


# Build the shared system messages once so later cells can reuse them.
SYSTEM_PROMPT = get_system_corpus("workplace")


# Guard the Python version: 3.14 or newer is too new for some course packages.
if sys.version_info[:2] >= (3, 14):
    raise RuntimeError(
        f"Python {sys.version.split()[0]} is too new for this course: some packages are not ready. "
        "Create the environment with Python 3.12 (3.11-3.13 work) and pick it as the notebook kernel."
    )

# Also stop on versions older than the supported 3.11.
assert (3, 11) <= sys.version_info[:2] <= (3, 13), (
    f"Use Python 3.11-3.13 (you have {sys.version.split()[0]})"
)
# Key lookup order:
#   1. file named in LM_ENV_FILE
#   2. ~/.langchain-mastery/.env (local, not synced)
#   3. .env next to notebooks.
# Only these four names are read from the file; everything else in it is ignored.

# Names read from the settings file, and folder names that suggest cloud-synced storage.
_ALLOWED = ("GROQ_API_KEY", "GROQ_MODEL", "LLM_MODE", "LM_ENV_FILE")
_SYNCED = ("onedrive", "sharepoint", "dropbox", "google drive", "googledrive", "icloud")


def _tilde(
    p,
):  # show the home folder as ~ so a shared screen does not show your user name
    p, home = str(p), str(Path.home())
    return (
        "~" + p[len(home) :]
        if p.lower().startswith(home.lower())
        else "(outside home) " + Path(p).name
    )


# Warn early if LM_ENV_FILE points to a file that does not exist.
_lm = os.getenv("LM_ENV_FILE", "").strip()
if _lm and not Path(_lm).is_file():
    print(
        "NOTE: LM_ENV_FILE is set but that file was not found, so it is ignored. Check the path."
    )

# Candidate settings files in lookup order; the first one that exists is used.
_env_files = [
    _lm,
    str(Path.home() / ".langchain-mastery" / ".env"),
    str(Path.cwd() / ".env"),
    str(Path.cwd() / "courseware" / ".env"),
]

# Copy only the allowed names from the chosen file into the environment, never overwriting values already set.
_env_file = next((p for p in _env_files if p and Path(p).is_file()), None)
if _env_file:
    try:
        from dotenv import dotenv_values

        for _k, _v in dotenv_values(_env_file).items():
            if (
                _k in _ALLOWED and _v is not None and _k not in os.environ
            ):  # existing variables win
                os.environ[_k] = _v

        # Tell the learner which file was used, with the home folder hidden.
        print(
            "Settings file:",
            _tilde(_env_file),
            "(only",
            ", ".join(_ALLOWED),
            "are read)",
        )

        # Warn if the file sits in a synced folder, where a key could be shared by accident.
        if any(w in _env_file.lower() for w in _SYNCED):
            print(
                "WARNING: your .env is in a synced folder (OneDrive, SharePoint, Dropbox, Google Drive "
                "or iCloud). Move it to ~/.langchain-mastery/.env"
            )

    except ImportError:
        print("python-dotenv not installed: using system environment variables only")
else:
    # With no settings file the notebook still runs in mock mode, so no key is needed.
    print("No .env file found: mock mode, no key needed.")

# Choose mock (offline, canned replies) or groq (live model); mock is the default.
LLM_MODE = os.getenv("LLM_MODE", "mock").strip().lower()
if LLM_MODE not in ("mock", "groq"):
    raise ValueError("LLM_MODE must be 'mock' or 'groq' (check your .env file)")

# Tracing switch: mock mode forces every tracing variable off; groq mode only warns if one is on.
_TRACING = ("LANGCHAIN_TRACING_V2", "LANGCHAIN_TRACING", "LANGSMITH_TRACING")
if LLM_MODE == "mock":
    for _t in _TRACING:
        os.environ[_t] = "false"

    print(
        "Mock mode: LangChain tracing is switched off for this session (nothing is sent anywhere)."
    )

elif any(os.getenv(_t, "").strip().lower() == "true" for _t in _TRACING):
    print(
        "WARNING: a LangChain tracing variable is on: live runs may be sent to a tracing service. Set it to false."
    )


class ScriptedToolChat(BaseChatModel):
    """Mock model for tool calling: replays a script. A str is a final answer; a dict
    {'name':..., 'args':{...}} (or a list of them) becomes AIMessage.tool_calls. Canned, not real."""

    # The script to replay and the position of the next step.
    script: list = []
    pos: int = 0

    @property
    def _llm_type(self):
        return "scripted-tool-chat"

    def bind_tools(self, tools, **kwargs):
        return self  # tools are ignored: the script decides

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        # Take the next scripted step, looping back to the start when the script runs out.
        step = (
            self.script[self.pos % len(self.script)] if self.script else "(mock reply)"
        )
        self.pos += 1

        # A text step becomes a final answer; a dict step becomes tool calls with made-up ids.
        if isinstance(step, str):
            msg = AIMessage(content=step)

        else:
            calls = step if isinstance(step, list) else [step]
            msg = AIMessage(
                content="",
                tool_calls=[
                    {
                        "name": c["name"],
                        "args": c.get("args", {}),
                        "id": f"mock_{self.pos}_{i}",
                        "type": "tool_call",
                    }
                    for i, c in enumerate(calls)
                ],
            )

        return ChatResult(generations=[ChatGeneration(message=msg)])


def get_llm(responses=None, tool_calls=None):
    """Return a chat model. Groq in 'groq' mode; offline canned mock otherwise."""

    if LLM_MODE == "groq":
        # Live mode needs both a model name and a key; otherwise explain what is missing.
        model = os.getenv("GROQ_MODEL", "").strip()

        if not model or not os.getenv("GROQ_API_KEY", "").strip():
            raise RuntimeError(
                "Groq mode needs GROQ_API_KEY and GROQ_MODEL in .env "
                "(suggested model: openai/gpt-oss-20b; confirm in the Groq console), "
                "or set LLM_MODE=mock."
            )

        # Return the Groq model; temperature 0 and a cache keep replies repeatable.
        print("LIVE model:", model)
        return ChatGroq(
            cache=InMemoryCache(),
            model=model,
            temperature=0,
            http_client=httpx.Client(verify=False),
        )

    # Mock mode: an offline fake model, with scripted tool calls if given, otherwise canned text replies.
    print("MOCK: canned replies")
    if tool_calls:
        return ScriptedToolChat(script=list(tool_calls))

    return FakeListChatModel(responses=list(responses or ["(mock reply)"]))


class TraceLog:
    """Tiny local trace: one dict per step. This class sends nothing anywhere."""

    RATE_PER_1K_TOKENS = 0.0005  # PLACEHOLDER rate in RM per 1,000 tokens (not a real price); use your plan's

    def __init__(self):
        self.rows = []

    def add(self, step, tool="-", tokens_in=0, tokens_out=0, seconds=0.0):
        # Estimate cost from the total tokens at the placeholder rate (not a real price).
        cost = (tokens_in + tokens_out) / 1000 * self.RATE_PER_1K_TOKENS

        # Store one row per step so the trace can be shown as a table.
        self.rows.append(
            {
                "step": step,
                "tool": tool,
                "tokens_in": tokens_in,
                "tokens_out": tokens_out,
                "seconds": round(seconds, 3),
                "est_cost_myr": round(cost, 6),
            }
        )

    def timed(self, step, fn, *args, tool="-", **kw):
        # Run fn, time it, and log a step; token counts are rough guesses (about 4 characters per token).
        t0 = time.perf_counter()
        out = fn(*args, **kw)

        self.add(
            step,
            tool,
            len(str(args)) // 4,
            len(str(out)) // 4,
            time.perf_counter() - t0,
        )

        return out

    # Sum tokens, seconds and estimated cost over all recorded steps.
    def total(self):
        return {
            k: round(sum(r[k] for r in self.rows), 6)
            for k in ("tokens_in", "tokens_out", "seconds", "est_cost_myr")
        }


class TfidfEmbeddings(Embeddings):
    """Offline embeddings: scikit-learn TF-IDF fitted on your corpus, L2-normalised."""

    def __init__(self, corpus):
        # Fit the vocabulary on the corpus once so documents and questions share one vector space.
        self.vec = TfidfVectorizer(norm="l2", lowercase=True).fit(list(corpus))

    def embed_documents(self, texts):
        return self.vec.transform(list(texts)).toarray().tolist()

    def embed_query(self, text):
        return self.embed_documents([text])[0]


# ---- END HELPER ----
# ---- END CANONICAL SETUP ----

No .env file found: mock mode, no key needed.
Mock mode: LangChain tracing is switched off for this session (nothing is sent anywhere).


## Three rules

1. **Keep your key private.** Never paste it into a notebook, chat or email.
2. **Do not paste personal or company data.** Live mode sends text to a third party (Groq). Use only this course's fictional data.
3. **AI output needs a human check.** A neat answer can still be wrong.

Mock mode (the default) needs no key.

**Practices:** a blank starter looks like `my_attempt = None`. **If you leave it as `None`, the notebook quietly runs the reference answer instead, so Run All works even when you wrote nothing.**

To learn, write your own and set `my_attempt` to it: then yours runs. `check(...)` prints ✔ or ✘ and a hint. `show_solution(step_id)` prints the reference code.

* `USE_SOLUTIONS`: `False` = you type the blanks,
* `USE_SOLUTIONS`: `True` = Run All reads the reference answers.

Reading first, then trying your own, teaches more.

In [4]:
# Notebook-specific imports and data path
import csv
import json
import re

import numpy as np
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

DATA = Path("data")
assert DATA is not None, "Data folder not found: open this notebook from courseware"

# Warn early if scikit-learn is missing, because the offline meaning map depends on it.
try:
    import sklearn

except ImportError:
    print("scikit-learn is needed: pip install scikit-learn, then restart the kernel")

# Confirm the mode and data folder before the lesson starts.
print("LLM_MODE =", LLM_MODE, "| data folder:", DATA)

LLM_MODE = mock | data folder: data


In [5]:
# Register how fast_forward rebuilds the Part A and Part B names for learners who join late.
rebuild("d1-04-A", "d1-04-A")
rebuild("d1-04-B", "d1-04-A", "d1-04-B")

In [6]:
# Keep the Part A loader as text so fast_forward can replay it without the learner's code.
_state_d1_04_a = r"""
def load_paragraphs(folder):
    docs, ids = [], []
    for f in sorted(folder.glob('sm-pol-*.txt')):
        sid = None
        for p in re.split(r'\n\s*\n', f.read_text(encoding='utf-8')):
            p = p.strip()
            if p.startswith('source_id:'):
                sid = p.split()[1]
            elif len(p) >= 40 and not p.startswith('Title:'):
                docs.append(p); ids.append(sid)
    return docs, ids
DOCS, IDS = load_paragraphs(DATA / 'workplace_corpus')
"""
state("d1-04-A", _state_d1_04_a)

# Part A reference code for the map, the search and the four questions with a person's picks.
state(
    "d1-04-A",
    "emb = TfidfEmbeddings(DOCS)\n"
    "DOC_VECS = np.array(emb.embed_documents(DOCS))\n"
    "def nearest(query, k=3):\n"
    "    sims = DOC_VECS @ np.asarray(emb.embed_query(query))\n"
    "    order = np.argsort(-sims)[:k]\n"
    "    return [(IDS[j], round(float(sims[j]), 3), DOCS[j]) for j in order]\n"
    "QUERIES = ['How many days of annual leave do I get?', 'Can I claim a taxi fare?',\n"
    "           'What is the limit for a hotel stay abroad?',\n"
    "           'Who do I call if the building alarm rings?']\n"
    "GOLD = ['SM-POL-01', 'SM-POL-03', 'SM-POL-08', 'SM-POL-04']",
)

# Reference answers below stand in for a practice left as None. Run All uses them.
ref(
    None,
    "d1-04-predict",
    "A person reads the topics and picks the policy.",
    "result = ['SM-POL-01', 'SM-POL-03', 'SM-POL-08', 'SM-POL-04']",
)

# Reference answer for explaining one wrong neighbour.
ref(
    None,
    "d1-04-why",
    "Everyday words match even when the topic differs.",
    "result = dict(why='shared common words', explain='The wrong paragraph shares everyday '\n"
    "              'words with the question, so it lands close on the map.')",
)

# Reference answer for the cosine similarity stretch.
ref(
    None,
    "d1-04-cosine",
    "Dot product divided by the two lengths.",
    "def cosine(a, b):\n"
    "    a, b = np.asarray(a), np.asarray(b)\n"
    "    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))\n"
    "result = cosine",
)

# Part B reference code that rebuilds the finished brief as a canned stand-in.
state(
    "d1-04-B",
    "with open(DATA / 'fifa_countries_audience.csv', newline='', encoding='utf-8') as f:\n"
    "    ROWS = list(csv.DictReader(f))\n"
    "TOP = sorted(ROWS, key=lambda r: -float(r['tv_audience_share']))[:3]\n"
    "NAMES = [r['country'] for r in TOP]\n"
    "TOTAL = round(sum(float(r['tv_audience_share']) for r in TOP), 1)\n"
    "brief = dict(headline='Three markets lead the TV audience', top_markets=NAMES,\n"
    "             top3_tv_share=TOTAL, recommendation='Plan staff viewing sessions first.')\n"
    "status = 'repaired'   # canned stand-in for the finished run",
)

# Reference answer for the written definition of done.
ref(
    None,
    "d1-04-done",
    "Words, a word limit and a call limit, written first.",
    "result = dict(sentence='The brief names the three biggest TV markets, gives their '\n"
    "              'combined share from the CSV, and stays short.',\n"
    "              max_words=40, max_calls=2)",
)

# Reference answer for the four self-check lines.
ref(
    None,
    "d1-04-reflect",
    "Four short, honest lines in your own words.",
    "result = dict(done='I wrote what a good brief looks like before I ran anything.',\n"
    "              check='The code compares the markets and total with the CSV.',\n"
    "              budget='The run makes at most two model calls, then stops.',\n"
    "              human='I will read the final brief and say what it leaves out.')",
)

# Reference answer for the budgeted generate-then-repair run.
ref(
    "d1-04-B",
    "d1-04-run",
    "The budget caps the calls; stop when clean.",
    "def run_brief(gen, fix, max_calls=2):\n"
    "    log, text, problems = [], '', []\n"
    "    for step in ('generate', 'repair')[:max_calls]:   # the budget caps the calls\n"
    "        chain, args = (gen, {'facts': FACTS}) if step == 'generate' else (\n"
    "            fix, {'bad': text, 'problems': '; '.join(problems)})\n"
    "        text = chain.invoke(args)\n"
    "        brief, problems = parse_and_check(text)\n"
    "        log.append(dict(step=step, problems=problems))\n"
    "        if not problems:\n"
    "            return brief, 'ok' if step == 'generate' else 'repaired', log\n"
    "    return None, 'budget' if max_calls < 2 else 'gave_up', log\n"
    "result = run_brief",
)

In [7]:
%pip install "langchain>=0.3,<0.4" "langchain-core>=0.3,<0.4" "langchain-groq>=0.3,<0.4" "langchain-text-splitters>=0.3,<0.4" "langgraph>=0.6,<0.7" "pydantic>=2,<3" "scikit-learn>=1.4" "python-dotenv"

In [8]:
import sys
import langchain
import langchain_core
import langchain_groq

print("Python:", sys.version)
print("LangChain:", getattr(langchain, "__version__", "unknown"))
print("LangChain Core:", getattr(langchain_core, "__version__", "unknown"))
print("LangChain Groq:", getattr(langchain_groq, "__version__", "unknown"))
print("LangChain file:", langchain.__file__)
print("Has verbose:", hasattr(langchain, "verbose"))

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
LangChain: 0.3.30
LangChain Core: 0.3.86
LangChain Groq: 0.3.8
LangChain file: /usr/local/lib/python3.13/dist-packages/langchain/__init__.py
Has verbose: True


In [9]:
print("DATA:", "DATA" in globals())
print("get_llm:", "get_llm" in globals())
print("pick:", "pick" in globals())
print("check:", "check" in globals())
print("USE_SOLUTIONS:", "USE_SOLUTIONS" in globals())

DATA: True
get_llm: True
pick: True
check: True
USE_SOLUTIONS: True


## The meaning map

| Policy id | Topic | Policy id | Topic |
|---|---|---|---|
| SM-POL-01 | annual leave | SM-POL-06 | data handling |
| SM-POL-02 | IT access | SM-POL-07 | onboarding |
| SM-POL-03 | expense claims | SM-POL-08 | business travel |
| SM-POL-04 | safety drill | SM-POL-09 | overtime |
| SM-POL-05 | remote work | SM-POL-10 | vendor onboarding |

### Idea: Text as a Point on a Map (●)

- An **embedding** turns a text into a **vector**: a list of numbers.

  Think of it as a pin on a map. Texts that use similar words land close together (see the picture at the top).

- **Cosine similarity** measures how close two pins point. With vectors of unit length 1, it is just the dot product.
- The **nearest neighbour** of a question is the policy note with the highest closeness number. A **chunk** here is one paragraph of a policy.
- Our vectors use **TF-IDF**: it counts words and gives rare words more weight. It runs offline, needs no model download, and finds shared *words*, not true *meaning*.

  **That is why the nearest note can be wrong.*

**Step 1 - Load the policy paragraphs (●)**

Each paragraph keeps the id of its policy file. Titles and very short lines are skipped.

**Run the cell as it is.*

In [10]:
def load_paragraphs(folder):  # one entry per paragraph, with its policy id
    docs, ids = [], []

    # Visit each policy file in order so every paragraph keeps the id of its own file.
    for f in sorted(folder.glob("sm-pol-*.txt")):
        sid = None

        # Split the file into paragraphs on blank lines.
        for p in re.split(r"\n\s*\n", f.read_text(encoding="utf-8")):
            p = p.strip()

            # The source_id line names the policy for the paragraphs that follow it.
            if p.startswith("source_id:"):
                sid = p.split()[1]

            # Keep real body paragraphs only: skip the title and very short lines.
            elif len(p) >= 40 and not p.startswith("Title:"):
                docs.append(p)
                ids.append(sid)

    return docs, ids


# Load the whole corpus: paragraph texts and, in the same order, their policy ids.
DOCS, IDS = load_paragraphs(DATA / "workplace_corpus")

# Check the load found paragraphs and policy files.
print(len(DOCS), "paragraphs from", len(set(IDS)), "policy files")

0 paragraphs from 0 policy files


**Step 2 - Build the map and the search (●)**

* `TfidfEmbeddings` gives each paragraph a vector.
* `nearest` rates every paragraph at once with a dot product and returns the best `k`.

Four questions and the policy a person would pick are set up for you.

In [11]:
from pathlib import Path

print("Current directory:", Path.cwd())
print("data folder exists:", Path("data").exists())
print("workplace_corpus exists:", Path("data/workplace_corpus").exists())

if Path("data/workplace_corpus").exists():
    files = list(Path("data/workplace_corpus").glob("sm-pol-*.txt"))
    print("Policy files found:", len(files))
    print("Files:", [f.name for f in files])

Current directory: /content
data folder exists: False
workplace_corpus exists: False


In [12]:
from pathlib import Path
import zipfile

zip_path = Path("/content/LLM.zip")
extract_path = Path("/content/llm_extracted")

print("ZIP exists:", zip_path.exists())

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

# Find the actual workplace_corpus folder
matches = list(extract_path.rglob("workplace_corpus"))

print("Corpus folders found:", len(matches))
for folder in matches:
    print("Corpus path:", folder)
    print("Policy files:", len(list(folder.glob("sm-pol-*.txt"))))

ZIP exists: True
Corpus folders found: 1
Corpus path: /content/llm_extracted/LLM/data/workplace_corpus
Policy files: 16


In [13]:
DOCS, IDS = load_paragraphs(DATA / "workplace_corpus")
print("Paragraphs:", len(DOCS))
print("Policy IDs:", len(set(IDS)))

Paragraphs: 0
Policy IDs: 0


In [14]:
from pathlib import Path

DATA = Path("/content/llm_extracted/LLM/data")

DOCS, IDS = load_paragraphs(DATA / "workplace_corpus")

print("Paragraphs:", len(DOCS))
print("Policy IDs:", len(set(IDS)))
print("Sample:", DOCS[0][:200] if DOCS else "No paragraphs loaded")

Paragraphs: 86
Policy IDs: 10
Sample: Seri Maju Berhad gives annual leave based on years of service. Staff with less than 2 years of service receive 14 days per calendar year. Staff with 2 to 5 years receive 16 days. Staff with more than 


In [15]:
# Turn every paragraph into a vector, one pin on the map.
emb = TfidfEmbeddings(DOCS)
DOC_VECS = np.array(emb.embed_documents(DOCS))

# Check the size of the map before searching it.
print("map size (paragraphs, vocabulary words):", DOC_VECS.shape)


# Search: score every paragraph against the question and return the k closest.
# Each result is (policy id, closeness, paragraph text).
def nearest(query, k=3):
    sims = DOC_VECS @ np.asarray(emb.embed_query(query))  # dot = cosine here

    # Rank from most to least similar and keep the top k.
    order = np.argsort(-sims)[:k]

    return [(IDS[j], round(float(sims[j]), 3), DOCS[j]) for j in order]


# Four everyday questions to try the search on.
QUERIES = [
    "How many days of annual leave do I get?",
    "Can I claim a taxi fare?",
    "What is the limit for a hotel stay abroad?",
    "Who do I call if the building alarm rings?",
]

GOLD = ["SM-POL-01", "SM-POL-03", "SM-POL-08", "SM-POL-04"]  # a person's picks

# Smoke test: one question, showing the policy id and closeness that come back.
print(nearest("I forgot my password", k=1)[0][:2])

map size (paragraphs, vocabulary words): (86, 522)
('SM-POL-02', 0.24)


**Predict first (●, no coding needed)**

Before you run the search, write which policy *you* would pick for each of the four questions above. Use the id table. Leave `my_attempt = None` to see an example.

In [16]:
my_pred = None  # replace None with four ids, e.g. ['SM-POL-01', 'SM-POL-02', ...]

# Use your predictions, or the example ones when my_pred is still None.
PRED = pick("d1-04-predict", my_pred, is_using_solutions=USE_SOLUTIONS)
# Check the picks have the right shape: four ids that look like SM-POL-03.
ok = len(PRED) == 4 and all(p.startswith("SM-POL-") and len(p) == 9 for p in PRED)

check("d1-04-predict", ok, "Write four ids in the form SM-POL-03.")

# Show the picks that the next cell compares with the search.
print(PRED)

✔ d1-04-predict: looks right
['SM-POL-01', 'SM-POL-03', 'SM-POL-08', 'SM-POL-04']


In [17]:
# Run the search for every question and keep only the single closest paragraph.
TOP1 = [nearest(q, k=1)[0] for q in QUERIES]  # (policy id, closeness, paragraph)
# Note which questions the search answered with a different policy from a person's pick.
WRONG = [i for i in range(4) if TOP1[i][0] != GOLD[i]]

# Side-by-side table: your guess, what the search found and what a person would pick.
print("question | your guess | search found | person would pick | result")

for q, p, t, g in zip(QUERIES, PRED, TOP1, GOLD):
    word = "same" if t[0] == g else "DIFFERENT"

    print(q[:30].ljust(32), p, t[0], g, word)

# Count the mismatches. The next cell looks into one of them.
print("Searches that found a different policy from the person:", len(WRONG))

question | your guess | search found | person would pick | result
How many days of annual leave    SM-POL-01 SM-POL-01 SM-POL-01 same
Can I claim a taxi fare?         SM-POL-03 SM-POL-03 SM-POL-03 same
What is the limit for a hotel    SM-POL-08 SM-POL-08 SM-POL-08 same
Who do I call if the building    SM-POL-04 SM-POL-04 SM-POL-04 same
Searches that found a different policy from the person: 0


**Explain one wrong neighbour (●)**

Look at the first question where the search found a different policy. The cell prints the question, the paragraph it found and the words they share.

Then choose a reason and write one sentence in your own words.

In [18]:
# Study the first mismatch, or the first question if there is none.
i = WRONG[0] if WRONG else 0
chosen = TOP1[i]
# Show the question and the paragraph the search picked for it.
print("question:       ", QUERIES[i])
print("wrong neighbour:", chosen[0], "closeness", chosen[1], "|", chosen[2][:70])
# Compare the two texts as sets of lowercase words.
words = lambda t: set(re.findall("[a-z]+", t.lower()))
# Show the words they share, which is what pulled the pin close.
print("shared words:   ", sorted(words(QUERIES[i]) & words(chosen[2])))
# Practice: choose a reason from the options and explain it in one sentence of your own.
OPTIONS = ["shared common words", "synonym not understood", "document too short"]
my_why = None  # replace None with: dict(why=<one of OPTIONS>, explain='your sentence')
# Use your answer, or the example one when my_why is None.
w = pick("d1-04-why", my_why, is_using_solutions=USE_SOLUTIONS)
# Self-check: a valid reason and a sentence of at least eight words.
check(
    "d1-04-why",
    w["why"] in OPTIONS and len(w["explain"].split()) >= 8,
    "Pick one of OPTIONS and write 8 or more words.",
)

question:        How many days of annual leave do I get?
wrong neighbour: SM-POL-01 closeness 0.275 | Medical leave is separate from annual leave. Staff must send a medical
shared words:    ['annual', 'leave', 'of']
✔ d1-04-why: looks right


True

### Check yourself

1. What is an embedding? <details><summary>Answer:</br></br></summary>A list of numbers that stands for a text, like a pin on a map.</br></br></details>
2. What does the nearest neighbour mean? <details><summary>Answer:</br></br></summary>The paragraph whose vector is closest to the question's vector.</br></br></details>
3. Why can the nearest paragraph be the wrong policy? <details><summary>Answer:</br></br></summary>TF-IDF matches shared words. Everyday words such as how, do and get can pull in a paragraph from another topic.</br></br></details>
4. What would you do before an answer cites that paragraph? <details><summary>Answer:</br></br></summary>Check the policy id and topic, and let a person confirm. A close pin is not proof.</br></br></details>

You might need `DOCS`, `IDS`, `emb` and `nearest`. The next cell tests it. If something is missing it rebuilds it from reference code (no API calls).

In [19]:
# Confirm the Part A names exist, for example if you joined late or skipped a cell.
need = ["DOCS", "IDS", "emb", "nearest", "QUERIES", "GOLD"]
missing = [n for n in need if n not in globals()]

print("All set for Part B." if not missing else "Missing: " + ", ".join(missing))

# Tell a late learner how to build the missing names.
if missing and "d1-04-A" not in CODES:
    print("Click the heading Part A, Run > Run All Above Selected Cell, then rerun.")

elif missing:
    fast_forward("d1-04-A")  # rebuilds the names above from reference code

# Run one search to confirm the map works before Part B.
hit = nearest("I forgot my password", k=1)[0][0]
print("nearest policy for a password question:", hit)

All set for Part B.
nearest policy for a password question: SM-POL-02


## The checked FIFA brief

### Idea: In one run (●)

The Corporate Affairs team plans a staff watch-party. They want a short brief:

* which three markets have the biggest TV audience share,
* their combined share, and
* one suggestion.

You will run it this way:

1. **Definition of done**, written first (in words, then as numbers).
2. **Structured answer** (a Pydantic shape) and a **facts check** against the CSV.
3. **One repair** at most, and a **call budget**.
4. **A human read** at the end: AI output needs a human check.

> ! **Data note** `fifa_countries_audience.csv` is public-style practice data about audience shares. It is used only to practise. Check the source before any real use.
>
> ! **MOCK warning** The first model reply below is canned and wrong on purpose (a wrong combined share), so you can watch the check catch it and the repair fix it. A live model will not follow this script.

**Step 1 - The facts and the shape (●)**

**Run this cell as it is.*

It reads the CSV, finds the top three markets by `tv_audience_share`, and defines the shape of a brief.

In [20]:
# Read the audience table from the CSV.
with open(DATA / "fifa_countries_audience.csv", newline="", encoding="utf-8") as f:
    ROWS = list(csv.DictReader(f))

# Find the three markets with the biggest TV audience share and their combined share.
TOP = sorted(ROWS, key=lambda r: -float(r["tv_audience_share"]))[:3]
NAMES = [r["country"] for r in TOP]
TOTAL = round(sum(float(r["tv_audience_share"]) for r in TOP), 1)
# Write the top three as a short facts sentence to give the model.
FACTS = "; ".join(
    r["country"] + " " + r["tv_audience_share"] + "% of TV audience" for r in TOP
)


# Define the shape of a brief. Pydantic limits catch a badly shaped reply, not wrong facts.
class Brief(BaseModel):
    headline: str = Field(max_length=80)
    top_markets: list[str] = Field(min_length=3, max_length=3)
    top3_tv_share: float = Field(ge=0, le=100)
    recommendation: str


# Show the facts the model will receive.
print(FACTS)

China 14.8% of TV audience; Brazil 7.1% of TV audience; Indonesia 6.7% of TV audience


**Step 2 - Write the definition of done (●, no coding needed)**

In one sentence, say what a good brief looks like. Then choose a word limit (20 to 60) and a call limit (2 or 3).

**Leave `my_attempt = None` to see an example.*

In [21]:
# Practice: write the definition of done first, before any model runs.
my_done = None  # replace None with: dict(sentence='...', max_words=40, max_calls=2)
# Use your definition, or the example one when my_done is None.
done = pick("d1-04-done", my_done, is_using_solutions=USE_SOLUTIONS)
# Turn the words into numbers the facts check can use: expected markets, total and limits.
DONE = dict(
    markets=NAMES, total=TOTAL, max_words=done["max_words"], max_calls=done["max_calls"]
)
# Self-check: a sentence of real length and limits inside the allowed ranges.
ok = (
    len(done["sentence"].split()) >= 8
    and 20 <= done["max_words"] <= 60
    and 2 <= done["max_calls"] <= 3
)
check("d1-04-done", ok, "Sentence of 8+ words; max_words 20-60; max_calls 2 or 3.")
print(DONE)

✔ d1-04-done: looks right
{'markets': ['China', 'Brazil', 'Indonesia'], 'total': 28.6, 'max_words': 40, 'max_calls': 2}


**Step 3 - The facts check and the prompts (●)**

+ Pydantic checks the *shape*.
+ `check_brief` checks the *facts* against the CSV and the word limit you chose.
+ `parse_and_check` joins them.

Run both cells as they are.

In [22]:
# A known-good brief, used to prove the check passes when the facts are right.
GOOD = dict(
    headline="Three markets lead the TV audience",
    top_markets=NAMES,
    top3_tv_share=TOTAL,
    recommendation="Plan staff viewing sessions first.",
)


def check_brief(b):  # the facts check: compare the brief with the CSV
    problems = []

    # The three markets must match the CSV, in any order.
    if sorted(b.top_markets) != sorted(DONE["markets"]):
        problems.append("top markets do not match the CSV")

    # The combined share must match the CSV total, allowing a small rounding gap.
    if abs(b.top3_tv_share - DONE["total"]) > 0.1:
        problems.append("combined share does not match the CSV")

    # The brief must stay within the word limit from the definition of done.
    if len((b.headline + " " + b.recommendation).split()) > DONE["max_words"]:
        problems.append("too many words")

    return problems


# Run the check on the good brief. It should find no problems.
print("GOOD brief problems:", check_brief(Brief(**GOOD)))

GOOD brief problems: []


In [23]:
def parse_and_check(text):  # shape first, then facts: (brief or None, problems)
    try:
        # Cut out the JSON object in case the model adds text around it.
        data = json.loads(text[text.find("{") : text.rfind("}") + 1])
        # Check the shape against Brief.
        brief = Brief.model_validate(data)

    # Report bad JSON or a wrong shape as a problem instead of crashing.
    except ValueError as err:
        return None, ["not valid JSON for Brief: " + str(err)[:60]]

    return brief, check_brief(brief)


# Prompt for the first call: ask for JSON only, with the facts filled in.
gen_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Reply with one JSON object only. Keys: headline, top_markets, "
            "top3_tv_share, recommendation. At most 40 words in total.",
        ),
        ("human", "Facts: {facts}"),
    ]
)

# Prompt for the repair call: send back the bad reply together with what the check found.
fix_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Fix the JSON so the check accepts it. Reply with JSON only."),
        ("human", "Bad reply: {bad} Check problems: {problems}"),
    ]
)

**Step 4 - the run with a budget (●)**

Complete the function. It makes at most `max_calls` model calls: first *generate*, then at most one *repair*. It stops as soon as the facts check is clean.

Its status is `ok`, `repaired`, `gave_up` or `budget`. The cell after it runs the brief with canned replies.

In [24]:
# Practice: build a run that generates a brief, then repairs it at most once, within a call budget.
def my_run(gen, fix, max_calls=2):
    # Keep the latest reply and its problems, plus a log of every call.
    log, text, problems = [], "", []

    for step in ("generate", "repair")[:max_calls]:  # the budget caps the calls
        # The first step asks the model for a brief. The repair step is yours to write.
        text = gen.invoke({"facts": FACTS}) if step == "generate" else text

        # TODO 1: for 'repair', ask fix.invoke({'bad': text, 'problems': ...})

        # Parse the reply and run the facts check on it.
        brief, problems = parse_and_check(text)
        log.append(dict(step=step, problems=problems))

        # TODO 2: no problems -> return brief, 'ok' or 'repaired', log

    # Fallback when no clean brief came out within the allowed calls.
    return None, "gave_up", log


# When your function is ready, assign it here. None keeps the reference version.
my_attempt = None  # when ready, write: my_attempt = my_run

run_brief = pick("d1-04-run", my_attempt, is_using_solutions=USE_SOLUTIONS)
print("run_brief is ready:", callable(run_brief))

run_brief is ready: True


In [25]:
# Canned replies for mock mode: a wrong first answer (inflated total) and a correct repair.
GEN_MOCK = [json.dumps({**GOOD, "top3_tv_share": round(TOTAL + 5, 1)})]  # canned: bad
FIX_MOCK = [json.dumps(GOOD)]  # canned: the repair is right


def make_chains():  # fresh canned models each time (canned replies replay in order)
    return (
        gen_prompt | get_llm(responses=GEN_MOCK) | StrOutputParser(),
        fix_prompt | get_llm(responses=FIX_MOCK) | StrOutputParser(),
    )


# Run the budgeted brief with the call limit from the definition of done, then show each step.
brief, status, log = run_brief(*make_chains(), max_calls=DONE["max_calls"])
for row in log:
    print(row["step"], "->", "; ".join(row["problems"]) or "no problems")

print("status:", status)

# Run again with a one-call budget to prove the budget really stops the run.
one = run_brief(*make_chains(), max_calls=1)
# Self-check: a clean result within budget, and the one-call run ends as 'budget'.
ok = status in ("ok", "repaired") and one[1] == "budget" and len(one[2]) == 1

# In mock mode the script is known: wrong first, then fixed by the repair.
if LLM_MODE == "mock":
    ok = ok and status == "repaired" and len(log) == 2

check("d1-04-run", ok, "At most max_calls calls; stop when clean; budget 1 -> budget.")

MOCK: canned replies
MOCK: canned replies
generate -> combined share does not match the CSV
repair -> no problems
status: repaired
MOCK: canned replies
MOCK: canned replies
✔ d1-04-run: looks right


True

### Self-check list (●)

Read the final brief below. Then, in your own words, write one line for each point. This is for you, not for anyone else. Leave `my_attempt = None` to see an example, then write your own and compare.

- **Done:** did I write what a good brief looks like before I ran it?
- **Check:** does code compare the brief with the CSV, not just its shape?
- **Budget:** does the run have a hard limit on model calls?
- **Human:** have I read the brief and said what it leaves out?

In [26]:
# Show the final brief for your own human read.
if brief is not None:
    print(brief.headline, "|", ", ".join(brief.top_markets), "|", brief.top3_tv_share)
    print(brief.recommendation)

# Practice: write one honest line in your own words for each self-check point.
my_notes = None  # replace None: dict(done='..', check='..', budget='..', human='..')

# Use your notes, or the example ones when my_notes is None.
notes = pick("d1-04-reflect", my_notes, is_using_solutions=USE_SOLUTIONS)

points = ("done", "check", "budget", "human")
# Self-check: every point has a few words, not a single word.
check(
    "d1-04-reflect",
    all(len(notes.get(k, "").split()) >= 4 for k in points),
    "Write 4 or more words for each of the four points.",
)

# Print your notes as a short list to compare with the example.
for k, v in notes.items():
    print(f"{k:<7}{v}")

Three markets lead the TV audience | China, Brazil, Indonesia | 28.6
Plan staff viewing sessions first.
✔ d1-04-reflect: looks right
done   I wrote what a good brief looks like before I ran anything.
check  The code compares the markets and total with the CSV.
budget The run makes at most two model calls, then stops.
human  I will read the final brief and say what it leaves out.


You should now have `NAMES`, `TOTAL`, `brief` and `status`.

In [27]:
# Confirm the Day 1 names exist before the wrap-up.
need = ["NAMES", "TOTAL", "brief", "status"]
missing = [n for n in need if n not in globals()]

print("Day 1 complete." if not missing else "Missing: " + ", ".join(missing))

if missing and not all(p in CODES for p in ("d1-04-A", "d1-04-B")):
    print("Click the heading Part A, Run > Run All Above Selected Cell, then rerun.")

elif missing:
    fast_forward("d1-04-B")  # rebuilds Part A and Part B names (canned, no API calls)

# One-line summary of the run for a final human read.
print("Top markets:", ", ".join(NAMES), "| total:", TOTAL, "| run status:", status)

Day 1 complete.
Top markets: China, Brazil, Indonesia | total: 28.6 | run status: repaired


## Recap and next step
- A **meaning map** turns text into vectors and finds the nearest note. The nearest note can still be wrong, so check it.
- The Day 1 recipe: write the **definition of done**, use a **structured answer** and a **facts check**, **repair once**, set a **call budget**, and **read the result yourself**.
- Mock replies are canned. Policies and amounts are fictional. AI output needs a human check.
- Next (Day 2): **[d2-01-policy-librarian-rag-and-citations.ipynb](d2-01-policy-librarian-rag-and-citations.ipynb)** (answer from the policies with citations, refuse when the answer is not there).

## STRETCH ◆ and TAKE-HOME ■ (only if you are on time, or after class)

### STRETCH ◆ - write cosine similarity yourself (about 6 min)
Cosine similarity is the dot product of two vectors divided by the product of their lengths. Write it, then compare two policy paragraphs.

In [28]:
# Stretch practice: write cosine similarity yourself.
def my_cosine(a, b):
    return None  # TODO: dot product divided by (length of a x length of b)


# Assign your function when ready. None keeps the reference version.
my_attempt = None  # when ready, write: my_attempt = my_cosine

cosine = pick("d1-04-cosine", my_attempt, is_using_solutions=USE_SOLUTIONS)

# Test on two easy cases: the same direction and a right angle.
same, far = cosine([1, 0], [1, 0]), cosine([1, 0], [0, 1])

ok = same is not None and far is not None and abs(same - 1) < 1e-9 and abs(far) < 1e-9
check("d1-04-cosine", ok, "Same direction gives 1; at right angles gives 0.")

# Apply it to two real paragraph vectors from the map.
val = cosine(DOC_VECS[0], DOC_VECS[1])
print("paragraph 0 against paragraph 1:", None if val is None else round(val, 3))

✔ d1-04-cosine: looks right
paragraph 0 against paragraph 1: 0.136


### Polish the brief (◆)

1. Change `max_words` to `25` in the written definition of done and run Step 4 again.

   Does the check still accept the canned repair?

2. Then add a fourth rule to `check_brief`, for example that the headline must name at least one country.

### Notes for later (■)

- **Other vectorisers.** `scikit-learn` also offers `CountVectorizer` and `HashingVectorizer`. Real meaning-based embedding models usually need large downloads (and often py`torch`), so they are not used in this course's low-spec setup.
- **Chart of the audience.** With `matplotlib`, draw a bar chart of `tv_audience_share` for the top 10 countries in `ROWS`, and label the three markets from your brief.
- **Honesty labels.** Mock replies are canned. Policies, amounts and sites are fictional. The FIFA table is public-style practice data: check the source before real use.